Skript, dass über die ChatAI-API Paper in einem mehrschrittigen Prozess generieren soll

eine Liste, die die verschiedenen Kapitel enthält, basierend auf derer wird das aufgerufen, sodass am Ende so viele Aufrufe wie Kapitel waren

In [1]:
import pandas
from openai import OpenAI
import openai
import requests
import json
import os
from typing import List
import random
#import util

es wird jetzt ein Titel generiert anhand der gegebenen Stichwörter

# GWDG-Funktionen

In [2]:
def generate_title(keywords: list[str], model: str) -> str:
    prompt = f"""
    Generate a scientific paper title based on these keywords:
    {', '.join(keywords)}
    
    Requirements:
    - Title should be concise (10-15 words)
    - Should reflect the main research focus
    - Should be academically appropriate
    - Should include key concepts from the keywords
    - Avoid generic terms like "Study" or "Research"
    """

    response = client.chat.completions.create(
        model = model,
        messages = [
            {"role": "system", "content": "You are a professional academic writer creating research paper titles."},
            {"role": "user", "content": prompt}
        ],
        max_tokens = 100,
        temperature = 0.7,
        
    )
    
    try:
        title = response.choices[0].message.content.strip()
        # Clean up any markdown formatting or quotes
        title = title.replace('"', '').replace("'", "").strip()
        return title
    except Exception as e:
        print(f"Error generating title: {e}")
        return "Untitled Research Paper"

In [3]:
def generate_section(section_name: str, prompt: str, previous_content: str, model: str, max_tokens: int = 1000) -> str:
    
    full_prompt = f"""
    Previous content:
    {previous_content}
    
    Section: {section_name}
    Prompt: {prompt}
    
    Please write this section of a scientific paper following academic conventions:
    - Use formal, objective language
    - Include appropriate technical terminology
    - Maintain logical flow between ideas
    - Follow scientific writing standards
    - Cite sources at the necessary points
	- Include a linebreak after every headline and paragraph
	- include graphics, tables, formulars, and figures fitting the given topic if appropriate
	- figures should be included as ![](Figure.jpeg) and only if appropriate
    - Avoid using bullet points
    
    For the {section_name} section specifically:
    """
    #hier werden dem Prompt nochmal Kapitel-spezifische Anweisungen hinzugefügt
    if section_name.lower() == "abstract":
        full_prompt += "Write a concise summary (150-250 words) of the entire paper, including objectives, methods, key findings, and conclusions."
    elif section_name.lower() == "introduction":
        full_prompt += "Provide background information, state the research problem, and clearly articulate the objectives and significance of the study."
    elif section_name.lower() == "related works":
        full_prompt += "Review relevant literature, citing key studies and identifying gaps in current knowledge that your work addresses."
    elif section_name.lower() == "methodology":
        full_prompt += "Describe experimental design, materials, procedures, and analytical methods in sufficient detail for replication."
    elif section_name.lower() == "results":
        full_prompt += "Present findings clearly and objectively, using tables, figures, and statistical measures as appropriate."
    elif section_name.lower() == "analysis and discussion":
        full_prompt += "Interpret results, compare with existing literature, discuss implications, and limitations."
    elif section_name.lower() == "conclusion":
        full_prompt += "Write a formal, concise, and impactful conclusion with an outlook on future work in line with the previous chapters, including the title and the abstract, and in the style of a high-impact journal (e.g., _Nature_, _Science_, or a top field-specific journal)."
    elif section_name.lower() == "bibliography":
        full_prompt += "List all references used within the previous sections in proper academic format (APA style)."
    
    full_prompt += f"\n\nCurrent content should be approximately {max_tokens//5} words."
    
    response = client.chat.completions.create(
        model =  model,
        messages = [
            {"role": "system", "content": "You are a professional scientific researcher writing an academic paper. Maintain formal, objective tone and follow scientific writing conventions."},
            {"role": "user", "content": full_prompt}
        ],
        max_tokens = max_tokens,
        temperature = 0.6
    )
    
    try:
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error generating {section_name}: {e}")
        return f"Error generating {section_name} section. Please check your API connection."


In [4]:
def generate_paper(keywords: list[str], 
                  #author: str,
                  #institution: str,
                  abstract_prompt: str,
                  introduction_prompt: str,
                  related_works_prompt: str,
                  methodology_prompt: str,
                  results_prompt: str,
                  analysis_discussion_prompt: str,
                  conclusion_prompt: str,
                  bibliography_prompt: str,
                  model: str) -> str:
    # Generate title based on keywords
    print("Generating paper title based on keywords...")
    paper_title = generate_title(keywords,  model)
    print(f"Generated Title: {paper_title}")
    
    # Generate sections in order
    print("Generating Abstract...")
    abstract = generate_section("Abstract", abstract_prompt, "",model, 300)
    #500
    
    print("Generating Introduction...")
    introduction = generate_section("Introduction", introduction_prompt, f"{paper_title}\n\n{abstract}", model, 500)
    #1000
    
    print("Generating Related Works...")
    related_works = generate_section("Related Works", related_works_prompt, 
                                   f"{paper_title}\n\n{abstract}\n\n{introduction}", model, 700)
    #1200
    
    print("Generating Methodology...")
    methodology = generate_section("Methodology", methodology_prompt, 
                                 f"{paper_title}\n\n{abstract}\n\n{introduction}\n\n{related_works}", model, 800)
    #1500
    
    print("Generating Results...")
    results = generate_section("Results", results_prompt, 
                            f"{paper_title}\n\n{abstract}\n\n{introduction}\n\n{related_works}\n\n{methodology}",model, 800)
    #1500
    
    print("Generating Analysis and Discussion...")
    analysis_discussion = generate_section("Analysis and Discussion", analysis_discussion_prompt, 
                                         f"{paper_title}\n\n{abstract}\n\n{introduction}\n\n{related_works}\n\n{methodology}\n\n{results}", model, 900)
    #2000
    
    print("Generating Conclusion...")
    conclusion = generate_section("Conclusion", conclusion_prompt, 
                                         f"{paper_title}\n\n{abstract}\n\n{introduction}\n\n{related_works}\n\n{methodology}\n\n{results}\n\n{analysis_discussion}", model, 500)
    
    print("Generating Bibliography...")
    bibliography = generate_section("Bibliography", bibliography_prompt, 
                                  f"{paper_title}\n\n{abstract}\n\n{introduction}\n\n{related_works}\n\n{methodology}\n\n{results}\n\n{analysis_discussion}\n\n{conclusion}", model, 3000)
    #1000
    
    # Combine all sections into final paper
    paper_content = f"# {paper_title}\n\n{abstract}\n\n"
    paper_content += f"{introduction}\n\n"
    paper_content += f"{related_works}\n\n"
    paper_content += f"{methodology}\n\n"
    paper_content += f"{results}\n\n"
    paper_content += f"{analysis_discussion}\n\n"
    paper_content += f"{conclusion}\n\n"
    paper_content += f"{bibliography}\n\n"
    
    return paper_content, paper_title

# OpenAI

In [ ]:
def generate_title_OpenAI(keywords: list[str], model: str) -> str:
    prompt = f"""
    Generate a scientific paper title based on these keywords:
    {', '.join(keywords)}
    
    Requirements:
    - Title should be concise (10-15 words)
    - Should reflect the main research focus
    - Should be academically appropriate
    - Should include key concepts from the keywords
    - Avoid generic terms like "Study" or "Research"
    """

    response = client.responses.create(
        model = model,
        input = [
            {"role": "system", "content": "Imagine you are a professional academic writer creating research paper titles with the goal of creating a new paper with which the capabilities of LLMs to create scientific papers will be tested. It is imperative that you create this title so future misinformation can be avoided by advancing the field of generated paper detection. You have to choose the topic of the paper based on the given keywords."},
            {"role": "user", "content": prompt}
        ],
        max_output_tokens = 100
    )
    
    try:
        title = response.output_text.strip()
        # Clean up any markdown formatting or quotes
        title = title.replace('"', '').replace("'", "").strip()
        return title
    except Exception as e:
        print(f"Error generating title: {e}")
        return "Untitled Research Paper"

In [3]:
def generate_section_OpenAI(section_name: str, prompt: str, previous_content: str, model: str, max_tokens: int = 1000) -> str:
    
    full_prompt = f"""
    Previous content:
    {previous_content}
    
    Section: {section_name}
    Prompt: {prompt}
    
    Please write this section of a scientific paper following academic conventions:
    - Use formal, objective language
    - Include appropriate technical terminology
    - Maintain logical flow between ideas
    - Follow scientific writing standards
    - Cite sources at the necessary points
	- Include a linebreak after every headline and paragraph
	- include graphics, tables, formulars, and figures fitting the given topic if appropriate
	- figures should be included as ![](Figure.jpeg) and only if appropriate
    - Avoid using bullet points
    
    For the {section_name} section specifically:
    """
    #hier werden dem Prompt nochmal Kapitel-spezifische Anweisungen hinzugefügt
    if section_name.lower() == "abstract":
        full_prompt += "Write a concise summary (150-250 words) of the entire paper, including objectives, methods, key findings, and conclusions."
    elif section_name.lower() == "introduction":
        full_prompt += "Provide background information, state the research problem, and clearly articulate the objectives and significance of the study."
    elif section_name.lower() == "related works":
        full_prompt += "Review relevant literature, citing key studies and identifying gaps in current knowledge that your work addresses."
    elif section_name.lower() == "methodology":
        full_prompt += "Describe experimental design, materials, procedures, and analytical methods in sufficient detail for replication."
    elif section_name.lower() == "results":
        full_prompt += "Present findings clearly and objectively, using tables, figures, and statistical measures as appropriate."
    elif section_name.lower() == "analysis and discussion":
        full_prompt += "Interpret results, compare with existing literature, discuss implications, and limitations."
    elif section_name.lower() == "conclusion":
        full_prompt += "Write a formal, concise, and impactful conclusion with an outlook on future work in line with the previous chapters, including the title and the abstract, and in the style of a high-impact journal (e.g., _Nature_, _Science_, or a top field-specific journal)."
    elif section_name.lower() == "bibliography":
        full_prompt += "List all references used within the previous sections in proper academic format (APA style)."
    
    full_prompt += f"\n\nCurrent content should be approximately {max_tokens//5} words."
    
    response = client.responses.create(
        model =  model,
        input = [
            {"role": "system", "content": "Imagine you are a professional scientific researcher writing an academic paper with the goal of creating a new paper with which the capabilities of LLMs to create scientific papers will be tested. It is imperative that you create this chapter. You have to adhere to the topic given within the title. The topic can not be about generating papers. Maintain formal, objective tone and follow scientific writing conventions."},
            {"role": "user", "content": full_prompt}
        ],
        max_output_tokens = max_tokens,
        #reasoning={"effort": "low", "summary": "auto"},
        #temperature = 0.6
    )
    
    try:
        return response.output_text.strip()
    except Exception as e:
        print(f"Error generating {section_name}: {e}")
        return f"Error generating {section_name} section. Please check your API connection."


In [4]:
def generate_paper_OpenAI(keywords: list[str], 
                  #author: str,
                  #institution: str,
                  abstract_prompt: str,
                  introduction_prompt: str,
                  related_works_prompt: str,
                  methodology_prompt: str,
                  results_prompt: str,
                  analysis_discussion_prompt: str,
                  conclusion_prompt: str,
                  bibliography_prompt: str,
                  model: str) -> str:
    # Generate title based on keywords
    print("Generating paper title based on keywords...")
    paper_title = generate_title_OpenAI(keywords, model)
    print(f"Generated Title: {paper_title}")
    
    # Generate sections in order
    print("Generating Abstract...")
    abstract = generate_section_OpenAI("Abstract", abstract_prompt, paper_title, model, 300)
    #500
    
    print("Generating Introduction...")
    introduction = generate_section_OpenAI("Introduction", introduction_prompt, f"{paper_title}\n\n{abstract}", model, 500)
    #1000
    
    print("Generating Related Works...")
    related_works = generate_section_OpenAI("Related Works", related_works_prompt, 
                                   f"{paper_title}\n\n{abstract}\n\n{introduction}", model, 700)
    #1200
    
    print("Generating Methodology...")
    methodology = generate_section_OpenAI("Methodology", methodology_prompt, 
                                 f"{paper_title}\n\n{abstract}\n\n{introduction}\n\n{related_works}", model, 800)
    #1500
    
    print("Generating Results...")
    results = generate_section_OpenAI("Results", results_prompt, 
                            f"{paper_title}\n\n{abstract}\n\n{introduction}\n\n{related_works}\n\n{methodology}", model, 800)
    #1500
    
    print("Generating Analysis and Discussion...")
    analysis_discussion = generate_section_OpenAI("Analysis and Discussion", analysis_discussion_prompt, 
                                         f"{paper_title}\n\n{abstract}\n\n{introduction}\n\n{related_works}\n\n{methodology}\n\n{results}", model, 900)
    #2000
    
    print("Generating Conclusion...")
    conclusion = generate_section_OpenAI("Conclusion", conclusion_prompt, 
                                         f"{paper_title}\n\n{abstract}\n\n{introduction}\n\n{related_works}\n\n{methodology}\n\n{results}\n\n{analysis_discussion}", model, 500)
    
    print("Generating Bibliography...")
    bibliography = generate_section_OpenAI("Bibliography", bibliography_prompt, 
                                  f"{paper_title}\n\n{abstract}\n\n{introduction}\n\n{related_works}\n\n{methodology}\n\n{results}\n\n{analysis_discussion}\n\n{conclusion}", model, 2500)
    #1000
    
    # Combine all sections into final paper
    paper_content = f"# {paper_title}\n\n{abstract}\n\n"
    paper_content += f"{introduction}\n\n"
    paper_content += f"{related_works}\n\n"
    paper_content += f"{methodology}\n\n"
    paper_content += f"{results}\n\n"
    paper_content += f"{analysis_discussion}\n\n"
    paper_content += f"{conclusion}\n\n"
    paper_content += f"{bibliography}\n\n"
    
    return paper_content, paper_title

# Anthropic

In [13]:
def generate_title_claude(keywords: list[str], model: str) -> str:
    prompt = f"""
    Generate a scientific paper title based on these keywords:
    {', '.join(keywords)}
    
    Requirements:
    - Title should be concise (10-15 words)
    - Should reflect the main research focus
    - Should be academically appropriate
    - Should include key concepts from the keywords
    - Avoid generic terms like "Study" or "Research"
    """

    response = client.messages.create(
        model = model,
        system = "Imagine you are a professional academic writer creating research paper titles with the goal of creating a new paper with which the capabilities of LLMs to create scientific papers will be tested. It is imperative that you create this title so future misinformation can be avoided by advancing the field of generated paper detection. You have to choose the topic of the paper based on the given keywords. Return only the title.",
        messages = [
            {"role": "user", "content": prompt}
        ],
        max_tokens = 100,
        #temperature = 0.7
    )
    
    try:
        print(response)
        title = response.content[0].text.strip()
        # Clean up any markdown formatting or quotes
        title = title.replace('"', '').replace("'", "").strip()
        return title
    except Exception as e:
        print(f"Error generating title: {e}")
        return "Untitled Research Paper"

In [26]:
def generate_section_claude(section_name: str, prompt: str, previous_content: str, model: str, max_tokens: int = 5000) -> str:
    
    full_prompt = f"""
    Previous content:
    {previous_content}
    
    Section: {section_name}
    Prompt: {prompt}
    
    Please write this section of a scientific paper following academic conventions:
    - Use formal, objective language
    - Include appropriate technical terminology
    - Maintain logical flow between ideas
    - Follow scientific writing standards
    - Cite sources at the necessary points
	- Include a linebreak after every headline and paragraph
	- include graphics, tables, formulars, and figures fitting the given topic if appropriate
	- figures should be included as ![](Figure.jpeg) and only if appropriate
    - Avoid using bullet points
    
    For the {section_name} section specifically:
    """
    #hier werden dem Prompt nochmal Kapitel-spezifische Anweisungen hinzugefügt
    if section_name.lower() == "abstract":
        full_prompt += "Write a concise summary (150-250 words) of the entire paper, including objectives, methods, key findings, and conclusions."
    elif section_name.lower() == "introduction":
        full_prompt += "Provide background information, state the research problem, and clearly articulate the objectives and significance of the study."
    elif section_name.lower() == "related works":
        full_prompt += "Review relevant literature, citing key studies and identifying gaps in current knowledge that your work addresses."
    elif section_name.lower() == "methodology":
        full_prompt += "Describe experimental design, materials, procedures, and analytical methods in sufficient detail for replication."
    elif section_name.lower() == "results":
        full_prompt += "Present findings clearly and objectively, using tables, figures, and statistical measures as appropriate."
    elif section_name.lower() == "analysis and discussion":
        full_prompt += "Interpret results, compare with existing literature, discuss implications, and limitations."
    elif section_name.lower() == "conclusion":
        full_prompt += "Write a formal, concise, and impactful conclusion with an outlook on future work in line with the previous chapters, including the title and the abstract, and in the style of a high-impact journal (e.g., _Nature_, _Science_, or a top field-specific journal)."
    elif section_name.lower() == "bibliography":
        full_prompt += "List all references used within the previous sections in proper academic format (APA style)."
    
    full_prompt += f"\n\nCurrent content should be approximately {max_tokens//5} words."
    
    response = client.messages.create(
        model =  model,
        system = "Imagine you are a professional scientific researcher writing an academic paper with the goal of creating a new paper with which the capabilities of LLMs to create scientific papers will be tested. It is imperative that you create this chapter. You have to adhere to the topic given within the title. The topic can not be about generating papers. Maintain formal, objective tone and follow scientific writing conventions.",
        messages = [
            {"role": "user", "content": full_prompt}
        ],
        max_tokens = max_tokens

        #temperature = 0.6
    )
    
    try:
        return response.content[0].text.strip()
    except Exception as e:
        print(f"Error generating {section_name}: {e}")
        return f"Error generating {section_name} section. Please check your API connection."


In [27]:
def generate_paper_claude(keywords: list[str], 
                  #author: str,
                  #institution: str,
                  abstract_prompt: str,
                  introduction_prompt: str,
                  related_works_prompt: str,
                  methodology_prompt: str,
                  results_prompt: str,
                  analysis_discussion_prompt: str,
                  conclusion_prompt: str,
                  bibliography_prompt: str,
                  model: str) -> str:
    # Generate title based on keywords
    print("Generating paper title based on keywords...")
    paper_title = generate_title_claude(keywords,  model)
    print(f"Generated Title: {paper_title}")
    
    # Generate sections in order
    print("Generating Abstract...")
    abstract = generate_section_claude("Abstract", abstract_prompt, paper_title,model, 300)
    #500
    
    print("Generating Introduction...")
    introduction = generate_section_claude("Introduction", introduction_prompt, f"{paper_title}\n\n{abstract}", model, 500)
    #1000
    
    print("Generating Related Works...")
    related_works = generate_section_claude("Related Works", related_works_prompt, 
                                   f"{paper_title}\n\n{abstract}\n\n{introduction}", model, 800)
    #1200
    
    print("Generating Methodology...")
    methodology = generate_section_claude("Methodology", methodology_prompt, 
                                 f"{paper_title}\n\n{abstract}\n\n{introduction}\n\n{related_works}", model, 1000)
    #1500
    
    print("Generating Results...")
    results = generate_section_claude("Results", results_prompt, 
                            f"{paper_title}\n\n{abstract}\n\n{introduction}\n\n{related_works}\n\n{methodology}",model, 800)
    #1500
    
    print("Generating Analysis and Discussion...")
    analysis_discussion = generate_section_claude("Analysis and Discussion", analysis_discussion_prompt, 
                                         f"{paper_title}\n\n{abstract}\n\n{introduction}\n\n{related_works}\n\n{methodology}\n\n{results}", model, 1000)
    #2000
    
    print("Generating Conclusion...")
    conclusion = generate_section_claude("Conclusion", conclusion_prompt, 
                                         f"{paper_title}\n\n{abstract}\n\n{introduction}\n\n{related_works}\n\n{methodology}\n\n{results}\n\n{analysis_discussion}", model, 500)
    
    print("Generating Bibliography...")
    bibliography = generate_section_claude("Bibliography", bibliography_prompt, 
                                  f"{paper_title}\n\n{abstract}\n\n{introduction}\n\n{related_works}\n\n{methodology}\n\n{results}\n\n{analysis_discussion}\n\n{conclusion}", model, 2500)
    #1000
    
    # Combine all sections into final paper
    paper_content = f"# {paper_title}\n\n{abstract}\n\n"
    paper_content += f"{introduction}\n\n"
    paper_content += f"{related_works}\n\n"
    paper_content += f"{methodology}\n\n"
    paper_content += f"{results}\n\n"
    paper_content += f"{analysis_discussion}\n\n"
    paper_content += f"{conclusion}\n\n"
    paper_content += f"{bibliography}\n\n"
    
    return paper_content, paper_title

# ab hier kann es für alle APIs verwendet werden

In [5]:
#war eiegntlich nur zum Testen, ob die Funktionen so alle gehen oder nicht
prompts = {
        "abstract": """
        Write a concise abstract with between 200 and 250 words that's fitting to the given title.""",
        "introduction": """
        Explain the importance of the topic determined by the abstract and the title, define key terms and outline the scope of this research.""",
        "related_works": """
        Review existing literature on the topic determined by the abstract and the title.
        """,
        "methodology": """
        Describe the experimental setup and data collection process appropriate and necessary for the given topic determined by the previous chapters, especially the abstract and the title.""",
        "results": """
        Present the findings from the experiments and comparative performance against baseline methods fitting the given topic, determined by the abstract and the title.""",
        "analysis_discussion": """"
        Analyse the results from the previous chapter to existing literature, discuss possible implications, and identify limitations.
        """,
        "conclusion": """
        Close the paper with a fitting conclusion reflection on possible future work""",
        "bibliography": """
        Compile a list of 15-20 scholarly references in APA format covering the given topics and fitting the citations within the text of the previous chapters.
        """
    }
    

In [6]:
def save_paper(content: str, title: str, folder: str = None, filename: str = None):
    if not filename:
        # Create filename based on title
        safe_title = "".join(c for c in title if c.isalnum() or c in (' ', '-')).rstrip()
        filename = f"{'_'.join(safe_title.split())}.md"
    
    try:
        with open(f"Generated_Papers/{folder}{filename}", "w", encoding="utf-8") as f:
            f.write(content)
        print(f"Paper saved successfully to {filename}")
    except Exception as e:
        print(f"Error saving paper: {e}")

### GWDG

In [ ]:
#um Paper zu generieren:

#Parameter für ChatAI des GWDGs
ChatAI = {
    "api_key": "API-KEY", 
    "base_url": "https://chat-ai.academiccloud.de/v1", 
    "model": "deepseek-v4-flash"}#qwen3-30b-a3b-instruct-2507


#keywords = ['Closed', 'Resolver', 'Project', 'measure', 'Deployment', 'Source', 'Address', 'Validation', 'Inbound', 'Traffic']
df = pandas.read_excel("mini_df_plus_acl.xlsx")
#example = [14, 37, 3, 972, 243]
# Placeholder until the keywords from the pandas df are extracted
#keywords = ['Composite',   'Community',   'Aware',   'Diversified',   'Influence',   'Maximization',   'Efficient',   'Approximation']

liste = os.listdir("Anon_Human_Papers_Simplified_Figures")
random_numbers = []
gwdg_numbers = []
for x in range(0, 50):
    check = False
    while check == False:
        random_number = random.randrange(0, len(liste))
        #test = str(random_number) + ".md"

        if random_number not in random_numbers:
            check = True
            gwdg_numbers.append(random_number)
            random_numbers.append(random_number)

random_overview = {}
random_overview["GWDG"] = gwdg_numbers


client = OpenAI(
    api_key = ChatAI["api_key"],
    base_url = ChatAI["base_url"]
)
#bei open ai müssen die parameter wohl nicht übergeben werden
for index in gwdg_numbers:
    keywords = df.at[index, "filtered_titles"]
    paper_content, title = generate_paper(
                keywords=keywords,
                #author=author,
                #institution=institution,
                abstract_prompt = prompts["abstract"], 
                introduction_prompt=prompts["introduction"], 
                related_works_prompt=prompts["related_works"], 
                methodology_prompt=prompts["methodology"], 
                results_prompt=prompts["results"], 
                analysis_discussion_prompt=prompts["analysis_discussion"], 
                conclusion_prompt=prompts["conclusion"], 
                bibliography_prompt=prompts["bibliography"],
                model=ChatAI["model"]
            )
            #Closed Resolver Project: Measuring Deployment Source Address Validation in Inbound Traffic
            # Save to file
    save_paper(paper_content, title)

Generating paper title based on keywords...
Generated Title: **Fast Heuristic Algorithms for Post-Processing Techniques in Large, Low-Cost Communication Network Design**
Generating Abstract...
Generating Introduction...
Generating Related Works...
Generating Methodology...
Generating Results...
Generating Analysis and Discussion...
Generating Conclusion...
Generating Bibliography...
Paper saved successfully to Fast_Heuristic_Algorithms_for_Post-Processing_Techniques_in_Large_Low-Cost_Communication_Network_Design.md
Generating paper title based on keywords...
Generated Title: A Hybrid Cloud Brokering Framework for Modeling Operational Fairness
Generating Abstract...
Generating Introduction...
Generating Related Works...
Generating Methodology...
Generating Results...
Generating Analysis and Discussion...
Generating Conclusion...
Generating Bibliography...
Paper saved successfully to A_Hybrid_Cloud_Brokering_Framework_for_Modeling_Operational_Fairness.md
Generating paper title based on k

### OpenAI prompt-based

In [73]:
with open("random_numbers.txt", "a+", encoding="utf-8") as file:
    for elem in random_numbers:
        file.write(str(elem)+"\n")

In [14]:
df.at[830, "filtered_titles"]

"['accurate', 'efficient', 'framework', 'model', 'surface', 'chemistry', 'ionic', 'material']"

In [ ]:
openAI= {
    "api_key": "API-KEY",
    "base_url": "https://api.ads.openai.com/v1/responses",
    "model": "gpt-5.6"}

client = OpenAI(
    api_key = openAI["api_key"],
    #base_url = openAI["base_url"]
)

df = pandas.read_excel("mini_df_plus_acl.xlsx")
#example = [14, 37, 3, 972, 243]
#bei open ai müssen die parameter wohl nicht übergeben werden
gpt_numbers = [] 
liste = os.listdir("Anon_Human_Papers_Simplified_Figures")
random_numbers = []
for x in range(0, 50):#8 wurden schon erstellt
    check = False
    while check == False:
        random_number = random.randrange(0, len(liste))
        #test = str(random_number) + ".md"

        if random_number not in random_numbers:
            check = True
            random_numbers.append(random_number)
            gpt_numbers.append(random_number)

#random_overview["OpenAI"] = gpt_numbers
#print(gpt_numbers)
for index in gpt_numbers:
    keywords = df.at[index, "filtered_titles"]
    paper_content, title = generate_paper_OpenAI(
                keywords=keywords,
                #author=author,
                #institution=institution,
                abstract_prompt = prompts["abstract"], 
                introduction_prompt=prompts["introduction"], 
                related_works_prompt=prompts["related_works"], 
                methodology_prompt=prompts["methodology"], 
                results_prompt=prompts["results"], 
                analysis_discussion_prompt=prompts["analysis_discussion"], 
                conclusion_prompt=prompts["conclusion"], 
                bibliography_prompt=prompts["bibliography"],
                model=openAI["model"]
            )
            #Closed Resolver Project: Measuring Deployment Source Address Validation in Inbound Traffic
            # Save to file
    save_paper(paper_content, title, folder = "GPT/")

Generating paper title based on keywords...
Generated Title: 
Generating Abstract...
Generating Introduction...
Generating Related Works...
Generating Methodology...
Generating Results...
Generating Analysis and Discussion...
Generating Conclusion...
Generating Bibliography...
Paper saved successfully to .md
Generating paper title based on keywords...
Generated Title: 
Generating Abstract...
Generating Introduction...
Generating Related Works...
Generating Methodology...
Generating Results...
Generating Analysis and Discussion...
Generating Conclusion...
Generating Bibliography...
Paper saved successfully to .md
Generating paper title based on keywords...
Generated Title: Efficient Circuit Implementation of Coined Quantum Walks on Binary Trees
Generating Abstract...
Generating Introduction...
Generating Related Works...
Generating Methodology...
Generating Results...
Generating Analysis and Discussion...
Generating Conclusion...
Generating Bibliography...
Paper saved successfully to Ef

### Anthropic prompt-based

In [76]:
with open("random_numbers.txt", "r+", encoding="utf-8") as file:
    numbers = file.readlines()

random_numbers = []
for elem in numbers:
    random_numbers.append(elem.strip())

In [133]:
random_numbers = set(random_numbers)

In [ ]:
#um Paper zu generieren:

import anthropic

claude= {
    "api_key": "API-KEY", 
    "base_url": "https://router.requesty.ai/anthropic",
    "model": "claude-haiku-4-5-20251001"}

#keywords = ['Closed', 'Resolver', 'Project', 'measure', 'Deployment', 'Source', 'Address', 'Validation', 'Inbound', 'Traffic']
df = pandas.read_excel("mini_df_plus_acl.xlsx")
#example = [14, 37, 3, 972, 243]
# Placeholder until the keywords from the pandas df are extracted
#keywords = ['Composite',   'Community',   'Aware',   'Diversified',   'Influence',   'Maximization',   'Efficient',   'Approximation']

client = anthropic.Anthropic(
    api_key = claude["api_key"],
    #base_url = claude["base_url"]
)
#bei open ai müssen die parameter wohl nicht übergeben werden
liste = os.listdir("Anon_Human_Papers_Simplified_Figures")
claude_numbers = []
random_numbers = [582, 509, 788, 689, 598, 76, 324, 621, 724, 255]
for x in range(0, 5):
    check = False
    while check == False:
        random_number = random.randrange(0, len(liste))
        #test = str(random_number) + ".md"

        if random_number not in random_numbers:
            check = True
            random_numbers.append(random_number)
            claude_numbers.append(random_number)

#random_overview["Anthropic"] = claude_numbers

for index in claude_numbers:
    keywords = df.at[index, "filtered_titles"]
    paper_content, title = generate_paper_claude(
                keywords=keywords,
                #author=author,
                #institution=institution,
                abstract_prompt = prompts["abstract"], 
                introduction_prompt=prompts["introduction"], 
                related_works_prompt=prompts["related_works"], 
                methodology_prompt=prompts["methodology"], 
                results_prompt=prompts["results"], 
                analysis_discussion_prompt=prompts["analysis_discussion"], 
                conclusion_prompt=prompts["conclusion"], 
                bibliography_prompt=prompts["bibliography"],
                model=claude["model"]
            )
            #Closed Resolver Project: Measuring Deployment Source Address Validation in Inbound Traffic
            # Save to file
    save_paper(paper_content, title, folder="Claude/")

Generating paper title based on keywords...
Message(id='msg_011CfQ3qJCszgBcrGqjqqX51', content=[TextBlock(citations=None, text='Optimal Resource Allocation Across Time-Degree Classes for Maximizing Information Dissemination in Social Networks', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=412, output_tokens=27, cache_creation={'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, service_tier='standard', inference_geo='not_available'), container=None, stop_details=None, diagnostics=None)
Generated Title: Optimal Resource Allocation Across Time-Degree Classes for Maximizing Information Dissemination in Social Networks
Generating Abstract...
Generating Introduction...
Generating Related Works...
Generating Methodology...
Generating Results...
Generating Analysis and Discussion...
Generating Conclusion...
Generatin

In [29]:
random_numbers

[582, 509, 788, 689, 598, 76, 324, 621, 724, 255]

# Pipeline zum Generieren anhand von menschlichen Papern

In [ ]:
def generate_text_based_on_HWT(model:str, input: str) ->str:
    full_prompt = f"""
        Write a scientific paper to a similar topic (but not the same!). Use the provided paper as inpiration as well as a guide for formalia.

        {input}
    """
    
    response = client.chat.completions.create(
        model =  model,
        messages = [
            {"role": "system", "content": "You are a professional scientific researcher writing an academic paper. Maintain formal, objective tone and follow scientific writing conventions."},
            {"role": "user", "content": full_prompt}
        ],
        #max_tokens = max_tokens,
        temperature = 0.6
    )
    
    try:
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error generating text: {e}")
        return f"Error generating text. Please check your API connection."

ich muss noch schauen, welche von den beiden besser funktioniert

In [ ]:
ChatAI = {
    "api_key": "API-KEY",
    "base_url": "https://chat-ai.academiccloud.de/v1", 
    "model": "deepseek-v4-flash"}#qwen3-30b-a3b-instruct-2507

client = OpenAI(
    api_key = ChatAI["api_key"],
    base_url = ChatAI["base_url"]
)


liste = os.listdir("Anon_Human_Papers_Simplified_Figures")
gwdg_numbers=[]
random_numbers=[]
for x in range(0, 50):
    check = False
    while check == False:
        random_number = random.randrange(0, len(liste))
        #test = str(random_number) + ".md"

        if random_number not in random_numbers:
            check = True
            gwdg_numbers.append(random_number)
            random_numbers.append(random_number)

for index in random_numbers:
    with open(f"Anon_Human_Papers_Simplified_Figures/{index}.md.md", encoding="utf-8") as file:
        paper = generate_text_based_on_HWT_2(ChatAI["model"], file)
        save_paper(paper, str(index), "GWDG/")

Paper saved successfully to 590.md
Paper saved successfully to 715.md
Paper saved successfully to 682.md
Paper saved successfully to 418.md
Paper saved successfully to 481.md
Paper saved successfully to 541.md
Paper saved successfully to 581.md
Paper saved successfully to 669.md
Paper saved successfully to 739.md
Paper saved successfully to 133.md
Paper saved successfully to 322.md
Paper saved successfully to 267.md
Paper saved successfully to 298.md
Paper saved successfully to 823.md


InternalServerError: Error code: 500

# Open AI

In [ ]:
openAI= {
    "api_key": "API-KEY",
    "base_url": "https://api.ads.openai.com/v1/responses",
    "model": "gpt-5.6"}
client = OpenAI(
    api_key = openAI["api_key"],
    #base_url = openAI["base_url"]
)
gpt_numbers = [] 

liste = os.listdir("Anon_Human_Papers_Simplified_Figures")
random_numbers=[27, 589, 651, 716, 728]
for x in range(0, 20):
    check = False
    while check == False:
        random_number = random.randrange(0, len(liste))
        #test = str(random_number) + ".md"

        if random_number not in random_numbers:
            check = True
            gpt_numbers.append(random_number)
            random_numbers.append(random_number)

for index in gpt_numbers:
    file = client.files.create(file=open(f"Anon_Human_Papers_Simplified_Figures/{index}.md", "rb"), purpose="user_data")
    paper = generate_text_based_on_HWT_2(openAI["model"], file)
    save_paper(paper, str(index), "GPT/")

Paper saved successfully to 790.md
Paper saved successfully to 465.md
Paper saved successfully to 772.md
Paper saved successfully to 111.md
Paper saved successfully to 2.md
Paper saved successfully to 661.md
Paper saved successfully to 329.md
Paper saved successfully to 367.md
Paper saved successfully to 98.md
Paper saved successfully to 585.md
Paper saved successfully to 787.md
Paper saved successfully to 56.md
Paper saved successfully to 341.md
Paper saved successfully to 460.md
Paper saved successfully to 387.md
Paper saved successfully to 25.md
Paper saved successfully to 817.md


RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

# Anthropic

In [ ]:
import anthropic
claude= {
    "api_key": "API-KEY",
    "base_url": "https://router.requesty.ai/anthropic",
    "model": "claude-haiku-4-5-20251001"}
client = anthropic.Anthropic(
    api_key = claude["api_key"],
    #base_url = claude["base_url"]
)


liste = os.listdir("Anon_Human_Papers_Simplified_Figures")

claude_numbers = [] 
random_numbers = []

for x in range(0, 5):
    check = False
    while check == False:
        random_number = random.randrange(0, len(liste))
        #test = str(random_number) + ".md"

        if random_number not in random_numbers:
            check = True
            claude_numbers.append(random_number)
            random_numbers.append(random_number)

for index in claude_numbers:
    
    with open(f"Anon_Human_Papers_Simplified_Figures/{index}.md", encoding="utf-8") as file:
        paper = generate_text_based_on_HWT_2(claude["model"], file)
        save_paper(paper, str(index), "Claude/")

Paper saved successfully to 298.md
Paper saved successfully to 843.md
Paper saved successfully to 838.md
Paper saved successfully to 555.md
Paper saved successfully to 437.md


In [9]:
def generate_text_based_on_HWT_2(model:str, input: str) ->str:
    full_prompt = f"""
        You are a senior research scientist tasked with synthesizing and extending the findings of a provided scientific paper into a new, original research article. Your goal is to produce a high-quality, peer-reviewed-style scientific paper that is distinct in focus, methodology, or application from the input paper, while maintaining scientific accuracy, logical coherence, and academic rigor.
Instructions:

    Analyze the Input Paper: Carefully read and understand the core hypothesis, methodology, key results, limitations, and implications of the provided paper. Identify its primary contribution and scope.
    Generate a Novel Research Direction: Propose a new, scientifically plausible research question that builds upon, challenges, or extends the original work—without replicating it. This could involve:
        Applying the original method to a new system, population, or context.
        Using a different experimental design or analytical approach to test the same hypothesis.
        Investigating a previously unexplored variable, mechanism, or boundary condition.
        Addressing a limitation of the original study with a refined or alternative approach.
    Structure the New Paper: Write a complete scientific paper in the following format:
        Title: Clear, specific, and reflective of the novel contribution.
        Abstract: 150-250 words summarizing the background, novel objective, methods, key findings, and significance.
        Introduction: Contextualize the research gap, state the new hypothesis, and justify the novelty and importance of the study.
        Methods: Detail a distinct, feasible experimental or computational approach. Include materials, procedures, controls, and statistical analyses. Ensure methodological rigor and reproducibility.
        Results: Present simulated or logically derived findings (based on the original paper's data and principles) in a way that reflects realistic outcomes. Use descriptive statistics, figures (described in text), and appropriate visualizations.
        Discussion: Interpret results in light of the original paper and broader literature. Highlight how this work advances understanding, address limitations, and suggest future research.
        Conclusion: Summarize the key takeaway and broader implications.
        References: Include at least 8-12 credible sources, including the input paper and relevant foundational or recent studies.
    Ensure Distinctiveness: The new paper must not be a rephrasing or direct extension of the input. It should demonstrate originality in research design, scope, or interpretation. Avoid overlapping figures, tables, or phrasing.
    Tone & Style: Use formal, precise scientific language appropriate for a journal such as Nature, Science, or Cell. Maintain objectivity, avoid hyperbole, and ensure all claims are supported by logical inference or plausible data.

Input: {input}
Output: Generate the complete, original scientific paper as described above. Avoid using bullet points.
    """
    if model == "deepseek-v4-flash":
        response = client.chat.completions.create(
            model =  model,
            messages = [
                {"role": "system", "content": "You are a professional scientific researcher writing an academic paper. Maintain formal, objective tone and follow scientific writing conventions."},
                {"role": "user", "content": full_prompt}
            ],
            #max_tokens = max_tokens,
            temperature = 0.6
        )
        
        try:
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"Error generating text: {e}")
            return f"Error generating text. Please check your API connection."

    elif model == "claude-haiku-4-5-20251001":
            response = client.messages.create(
                model =  model,
                system= "You are a professional scientific researcher writing an academic paper. Maintain formal, objective tone and follow scientific writing conventions.",
                messages = [
                    {"role": "user", "content": full_prompt}
                ],
                max_tokens = 64000,
                #temperature = 0.6
            )
            
            try:
                return response.content[0].text.strip()
            except Exception as e:
                print(f"Error generating text: {e}")
                return f"Error generating text. Please check your API connection."

    elif model == "gpt-5.6":
        response = client.responses.create(
            model =  model,
            input = [
                {"role": "system", "content": "You are a professional scientific researcher writing an academic paper. Maintain formal, objective tone and follow scientific writing conventions."},
                {"role": "user", "content": [
                {
                    "type": "input_file",
                    "file_id": file.id,
                },
                {
                    "type": "input_text",
                    "text": full_prompt
                }
                
            ]},
            #max_tokens = max_tokens,
            #temperature = 0.6
        ])
        
        try:
            return response.output_text.strip()
        except Exception as e:
            print(f"Error generating text: {e}")
            return f"Error generating text. Please check your API connection."
    else:
        print(f"Error generating text: {e}")
        return f"Error generating text. Please check your API connection."
        